## Import cấu hình và đường dẫn input/output


In [18]:
# Import và cấu hình đường dẫn input/output
from pathlib import Path
import pandas as pd
import numpy as np
import unicodedata, re

candidates_in = [
    Path("../data/extracted/mogi_raw.csv")
]
IN_PATH = next((p for p in candidates_in if p.exists()), candidates_in[-1])

OUT_PATH = (IN_PATH.parent.parent / "preprocessed" / "mogi_preprocessed.csv")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Input:  {IN_PATH.resolve()}")
print(f"Output: {OUT_PATH.resolve()}")

Input:  C:\Users\ADMIN\source\Năm 3 - HK1\Nhập môn KHDL\project\data\extracted\mogi_raw.csv
Output: C:\Users\ADMIN\source\Năm 3 - HK1\Nhập môn KHDL\project\data\preprocessed\mogi_preprocessed.csv


## Đọc CSV và làm sạch tên cột/chuỗi


In [19]:
# Đọc CSV (xử lý BOM, bỏ dòng lỗi nếu có)
df = pd.read_csv(
    IN_PATH,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
    encoding_errors="ignore",
    on_bad_lines="skip"
)

# Bỏ BOM trong header + strip khoảng trắng tên cột
df.columns = [c.replace("\ufeff", "").strip() for c in df.columns]

# Trim khoảng trắng toàn bộ chuỗi
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

print("Kích thước ban đầu:", df.shape)
display(df.head())

Kích thước ban đầu: (2774, 9)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_118144\492557534.py:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


,tieu_de,gia,dia_chi,dien_tich_dat,phong_ngu,phong_tam,so_tang,phap_ly,ngay_dang
0,Quận 1 - Nhà Mới Ở Ngay - 25M - 2PN - Nhỉnh 3 Tỷ,3.9,"Bùi Viện, Phường Phạm Ngũ Lão, Quận 1, TPHCM",25.0,2,1,2,Sổ hồng,25/11/2025
1,📣 bùi viện quận 1 - nhà mới đẹp ở ngay - phù h...,8.6,"Bùi Viện, Phường Phạm Ngũ Lão, Quận 1, TPHCM",25.0,2,,3,Sổ đỏ,25/11/2025
2,Bùi Viện Quận 1 - Nhà Mới Đẹp - Phù Hợp Ở Hoặc...,8.6,"Bùi Viện, Phường Phạm Ngũ Lão, Quận 1, TPHCM",25.0,2,3,3,Sổ hồng,25/11/2025
3,Bán nhà cũ tiện xây mới MT Võ Thị Sáu Q.1 - 20...,47.0,"Võ Thị Sáu, Phường Tân Định, Quận 1, TPHCM",206.0,1,1,18,Sổ hồng,25/11/2025
4,Bán nhà hxh Nguyễn Cảnh Chân - Trần Đình Xu - ...,7.6,"Trần Đình Xu, Phường Nguyễn Cư Trinh, Quận 1, ...",30.0,6,5,17,Sổ hồng,25/11/2025


## Loại bỏ mọi dòng có giá trị thiếu


In [20]:
missing_like = re.compile(r"^(?:\s*|na|n/a|none|null|nan)$", flags=re.IGNORECASE)
df = df.replace(missing_like, np.nan, regex=True)

n0 = len(df)
df = df.dropna(how="any")
print(f"- Drop thiếu (any column): {n0} -> {len(df)}")

- Drop thiếu (any column): 2774 -> 2088


## Loại bỏ dòng bị trùng lặp

In [21]:
# Dedup exact theo 2 cột: tieu_de + dia_chi (không chuẩn hóa)
required = ["tieu_de", "dia_chi"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Thiếu cột để dedup: {missing}")

before = len(df)
dup_exact = df.duplicated(subset=required, keep="first").sum()
print(f"Số dòng duplicate (exact match theo {required}): {dup_exact}")

df = df.drop_duplicates(subset=required, keep="first").copy()
after = len(df)
print(f"- Kết quả dedup: {before} -> {after} (đã bỏ {before - after} dòng)")

try:
    display(df.head())
except:
    pass

Số dòng duplicate (exact match theo ['tieu_de', 'dia_chi']): 9
- Kết quả dedup: 2088 -> 2079 (đã bỏ 9 dòng)


,tieu_de,gia,dia_chi,dien_tich_dat,phong_ngu,phong_tam,so_tang,phap_ly,ngay_dang
0,Quận 1 - Nhà Mới Ở Ngay - 25M - 2PN - Nhỉnh 3 Tỷ,3.9,"Bùi Viện, Phường Phạm Ngũ Lão, Quận 1, TPHCM",25.0,2,1,2,Sổ hồng,25/11/2025
2,Bùi Viện Quận 1 - Nhà Mới Đẹp - Phù Hợp Ở Hoặc...,8.6,"Bùi Viện, Phường Phạm Ngũ Lão, Quận 1, TPHCM",25.0,2,3,3,Sổ hồng,25/11/2025
3,Bán nhà cũ tiện xây mới MT Võ Thị Sáu Q.1 - 20...,47.0,"Võ Thị Sáu, Phường Tân Định, Quận 1, TPHCM",206.0,1,1,18,Sổ hồng,25/11/2025
4,Bán nhà hxh Nguyễn Cảnh Chân - Trần Đình Xu - ...,7.6,"Trần Đình Xu, Phường Nguyễn Cư Trinh, Quận 1, ...",30.0,6,5,17,Sổ hồng,25/11/2025
6,"MẶT TIỀN Trần Quang Khải + Hai Bà Trưng, QUẬN ...",99.0,"Trần Quang Khải, Phường Tân Định, Quận 1, TPHCM",398.0,2,2,9,Sổ hồng,25/11/2025


## Kiểm tra số: phòng ngủ, phòng tắm, số tầng



In [22]:
# Kiểm tra numeric cho các cột: int (phong_ngu/phong_tam/so_tang) và float (gia/dien_tich_dat)

# 1) Các cột số nguyên
int_cols = ["phong_ngu", "phong_tam", "so_tang"]
present_int = [c for c in int_cols if c in df.columns]

# Ép numeric (NaN nếu không hợp lệ)
for c in present_int:
    df[c] = pd.to_numeric(df[c], errors="coerce")

n_before_int = len(df)
if present_int:
    df = df.dropna(subset=present_int)
    # Loại giá trị âm (>= 0). Nếu muốn loại cả 0 thì đổi thành > 0
    cond_nonneg = np.ones(len(df), dtype=bool)
    for c in present_int:
        cond_nonneg &= df[c] >= 0
    df = df[cond_nonneg]

# Ép kiểu Int64 (nullable)
for c in present_int:
    df[c] = df[c].astype("Int64")

print(f"- Lọc numeric INT {present_int}: {n_before_int} -> {len(df)}")

# 2) Các cột số thực
float_cols = ["gia", "dien_tich_dat"]
present_float = [c for c in float_cols if c in df.columns]

def to_float_clean(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if not s:
        return np.nan
    # chuẩn hóa thập phân: , -> .
    s = s.replace(",", ".")
    # bỏ mọi ký tự không phải số, dấu chấm hoặc dấu âm (loại 'm2', 'm²', ' m ')
    s = re.sub(r"[^0-9.\-]", "", s)
    # fallback: nếu vẫn lỗi, lấy số đầu tiên xuất hiện
    try:
        return float(s)
    except:
        m = re.search(r"-?\d+(?:\.\d+)?", s)
        return float(m.group(0)) if m else np.nan

for c in present_float:
    df[c] = df[c].apply(to_float_clean)

n_before_float = len(df)
if present_float:
    df = df.dropna(subset=present_float)
    # Không âm; nếu muốn loại cả 0 thì đổi > 0
    cond_nonneg = np.ones(len(df), dtype=bool)
    for c in present_float:
        cond_nonneg &= df[c] >= 0
    df = df[cond_nonneg]

print(f"- Lọc numeric FLOAT {present_float}: {n_before_float} -> {len(df)}")

# Kiểm tra dtype kết quả (tùy chọn)
print(df.dtypes)

- Lọc numeric INT ['phong_ngu', 'phong_tam', 'so_tang']: 2079 -> 2079
- Lọc numeric FLOAT ['gia', 'dien_tich_dat']: 2079 -> 2079
tieu_de           object
gia              float64
dia_chi           object
dien_tich_dat    float64
phong_ngu          Int64
phong_tam          Int64
so_tang            Int64
phap_ly           object
ngay_dang         object
dtype: object


## Chuẩn hóa cột pháp lý 

In [23]:
def normalize_legal(val: str) -> str:
    s = str(val or "").strip()
    # Bỏ dấu + thường hóa
    s_norm = unicodedata.normalize("NFKD", s)
    s_norm = "".join(ch for ch in s_norm if not unicodedata.combining(ch)).lower()
    s_norm = re.sub(r"[^a-z0-9]+", " ", s_norm).strip()
    if "so do" in s_norm:
        return "Sổ đỏ"
    if "so hong" in s_norm:
        return "Sổ hồng"
    return "Giấy tờ không xác định"

if "phap_ly" in df.columns:
    df["phap_ly"] = df["phap_ly"].apply(normalize_legal)

display(df[["phap_ly"]].head() if "phap_ly" in df.columns else df.head())

,phap_ly
0,Sổ hồng
2,Sổ hồng
3,Sổ hồng
4,Sổ hồng
6,Sổ hồng


## Chuẩn hóa ngày đăng về date-time


In [24]:
if "ngay_dang" in df.columns:
    df["ngay_dang"] = pd.to_datetime(df["ngay_dang"], dayfirst=True, errors="coerce")

display(df[["ngay_dang"]].head() if "ngay_dang" in df.columns else df.head())

,ngay_dang
0,2025-11-25
2,2025-11-25
3,2025-11-25
4,2025-11-25
6,2025-11-25


## Lưu ra file

In [25]:
before_final = len(df)
df = df.dropna(how="any")
print(f"- Drop thiếu lần cuối (sau chuẩn hóa): {before_final} -> {len(df)}")

df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"Đã lưu: {OUT_PATH} ({len(df)} dòng)")
display(df.head())

- Drop thiếu lần cuối (sau chuẩn hóa): 2079 -> 2079
Đã lưu: ..\data\preprocessed\mogi_preprocessed.csv (2079 dòng)


,tieu_de,gia,dia_chi,dien_tich_dat,phong_ngu,phong_tam,so_tang,phap_ly,ngay_dang
0,Quận 1 - Nhà Mới Ở Ngay - 25M - 2PN - Nhỉnh 3 Tỷ,3.9,"Bùi Viện, Phường Phạm Ngũ Lão, Quận 1, TPHCM",25.0,2,1,2,Sổ hồng,2025-11-25
2,Bùi Viện Quận 1 - Nhà Mới Đẹp - Phù Hợp Ở Hoặc...,8.6,"Bùi Viện, Phường Phạm Ngũ Lão, Quận 1, TPHCM",25.0,2,3,3,Sổ hồng,2025-11-25
3,Bán nhà cũ tiện xây mới MT Võ Thị Sáu Q.1 - 20...,47.0,"Võ Thị Sáu, Phường Tân Định, Quận 1, TPHCM",206.0,1,1,18,Sổ hồng,2025-11-25
4,Bán nhà hxh Nguyễn Cảnh Chân - Trần Đình Xu - ...,7.6,"Trần Đình Xu, Phường Nguyễn Cư Trinh, Quận 1, ...",30.0,6,5,17,Sổ hồng,2025-11-25
6,"MẶT TIỀN Trần Quang Khải + Hai Bà Trưng, QUẬN ...",99.0,"Trần Quang Khải, Phường Tân Định, Quận 1, TPHCM",398.0,2,2,9,Sổ hồng,2025-11-25
